In [1]:
text="hello"
tokens=list(text)
print(tokens)

['h', 'e', 'l', 'l', 'o']


# Chapter 2 — Tokenization & Embeddings

## LLM Learning From Scratch

In this chapter, we will understand:

- What tokenization is
- Character-level tokenization
- Word-level tokenization
- Subword tokenization
- Vocabulary
- Token IDs
- Encoding and decoding
- Token embeddings
- Embedding dimensions
- Positional information

The goal is to understand how raw text is converted into numerical
representations that a neural network can process.


# 1. Why Do We Need Tokenization?

Computers and neural networks work with numbers.

But human language is text.

For example:

"I love machine learning"

cannot be directly passed into a neural network.

We first need to convert the text into smaller pieces called **tokens**.

Example:

"I love machine learning"

↓

["I", "love", "machine", "learning"]

These tokens can then be mapped to numerical IDs.

↓

[0, 1, 2, 3]

This process is called **tokenization**.

In [2]:
text = "hello"

tokens = list(text)

print(tokens)

['h', 'e', 'l', 'l', 'o']


# 2. Character-Level Tokenization

Character-level tokenization treats every character as a token.

Example:

"hello"

↓

["h", "e", "l", "l", "o"]

### Advantages

- Very small vocabulary
- Can represent almost any text

### Disadvantages

- Very long sequences
- The model has to learn words from individual characters
- Less efficient for language modeling

Modern LLMs generally do not use pure character-level tokenization.

In [3]:
text="I love machine learning"

tokens=text.split()

print(tokens)

['I', 'love', 'machine', 'learning']


# 3. Word-Level Tokenization

Here, each word becomes a token.

"I love machine learning"

↓

["I", "love", "machine", "learning"]

This is easier to understand than character-level tokenization.

However, it creates another problem.

Consider:

"play"

"playing"

"played"

"player"

These are related words, but word-level tokenization treats them as
completely separate tokens.

There can also be an enormous number of unique words.

Therefore, modern LLMs generally use **subword tokenization**.

In [4]:
text="I love machine learning and I love Python"

tokens=text.split()

vocabulary=sorted(set(tokens))

print("Vocabulary:")
for i,token in enumerate(vocabulary):
  print(i,token)

Vocabulary:
0 I
1 Python
2 and
3 learning
4 love
5 machine


# 4. Vocabulary

A vocabulary is a collection of all tokens known by the tokenizer.

For example:

```text
Vocabulary

0 → I
1 → Python
2 → and
3 → learning
4 → love
5 → machine

In [6]:
token_to_id={
    token:i
    for i, token in enumerate(vocabulary)
}

print(token_to_id)

encoded=[token_to_id[token] for token in tokens]

print(tokens)
print(encoded)

{'I': 0, 'Python': 1, 'and': 2, 'learning': 3, 'love': 4, 'machine': 5}
['I', 'love', 'machine', 'learning', 'and', 'I', 'love', 'Python']
[0, 4, 5, 3, 2, 0, 4, 1]


In [7]:
from IPython.core.inputtransformer2 import TokenTransformBase
id_to_token={
    i:token
    for token, i in token_to_id.items()
}

decoded_tokens=[id_to_token[i] for i in encoded]

decoded_text=" ".join(decoded_tokens)

print(decoded_text)

I love machine learning and I love Python


# 5. The Unknown Token Problem

Suppose our vocabulary contains:

["I", "love", "Python"]

Now we receive:

"I love Transformers"

But "Transformers" isn't in our vocabulary.

What do we do?

One traditional solution is an `<UNK>` token.

```text
"I love Transformers"

↓

["I", "love", "<UNK>"]




# 6. Subword Tokenization

Instead of requiring every complete word to exist in the vocabulary,
we break words into smaller pieces.

For example, a tokenizer might represent:

"playing"

as:

["play", "ing"]

and:

"unhappiness"

as something like:

["un", "happiness"]

The exact splitting depends on the tokenizer.

The important idea is:

> A word can be represented using smaller pieces that already exist
> in the vocabulary.

This gives modern LLMs a useful balance:

Character-level:
- Small vocabulary
- Very long sequences

Word-level:
- Large vocabulary
- Shorter sequences

Subword-level:
- Reasonable vocabulary
- Reasonable sequence length
- Can handle unseen or uncommon words

In [8]:
!pip -q install transformers

In [9]:
from transformers import AutoTokenizer

tokenizer=AutoTokenizer.from_pretrained("gpt2")

text="I love machine learning"
tokens=tokenizer.tokenize(text)
print(tokens)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

['I', 'Ġlove', 'Ġmachine', 'Ġlearning']


In [10]:
token_ids=tokenizer.encode(text)
print(token_ids)

[40, 1842, 4572, 4673]


In [11]:
decoded=tokenizer.decode(token_ids)
print(decoded)

I love machine learning


# 7. Why Token IDs Are Not Enough

Suppose we have:

cat → 10
dog → 25
car → 73

These numbers are only identifiers.

The model should NOT interpret:

dog = 25

as meaning that a dog is somehow "larger" or "more important" than a cat
because 25 > 10.

Token IDs are simply labels.

We need a better representation.

This is where embeddings come in.


# 8. Token Embeddings

An embedding represents a token as a vector of numbers.

Instead of:

cat → 10

we might have:

cat → [0.21, -0.43, 0.87, 0.15]

dog → [0.19, -0.40, 0.82, 0.18]

car → [-0.72, 0.31, 0.14, -0.55]

Each token gets a vector.

These vectors are learned during training.

The embedding dimensions don't have simple predefined meanings like:

dimension 1 = animal

dimension 2 = size

Instead, the model learns useful representations from data.

In [12]:
import torch
import torch.nn as nn

vocab_size=10
embedding_dim=4

embedding=nn.Embedding(vocab_size, embedding_dim)

print(embedding.weight)

Parameter containing:
tensor([[-0.5590,  0.1940,  1.6508,  2.1351],
        [ 0.3704, -0.9041,  1.7927,  0.4268],
        [ 0.3278,  0.6487,  0.5254, -1.5987],
        [-1.9767, -0.0304, -1.0680,  0.2554],
        [-0.6373,  0.8912,  0.0279, -0.4165],
        [-1.1906, -1.2446,  0.2988, -0.9938],
        [ 0.2342, -0.4178, -0.7240, -0.4756],
        [-0.5131, -1.4881,  0.3858, -0.3422],
        [ 0.9596,  1.0700,  1.1510, -1.2061],
        [-1.7494, -1.3293,  1.8037,  0.1173]], requires_grad=True)


In [13]:
token_id=torch.tensor([3])
vector=embedding(token_id)
print("Token ID:", token_id)
print("Embedding:",vector)
print("Shape:",vector.shape)

Token ID: tensor([3])
Embedding: tensor([[-1.9767, -0.0304, -1.0680,  0.2554]], grad_fn=<EmbeddingBackward0>)
Shape: torch.Size([1, 4])


The embedding layer can be viewed as a matrix:

                 Embedding Dimensions
                 ↓
        0.21   -0.43   0.87   0.15
        0.19   -0.40   0.82   0.18
       -0.72    0.31   0.14  -0.55
        ...
        
        ↑
     Token IDs

Each row represents one token.

If the vocabulary contains V tokens and the embedding dimension is D:

Embedding matrix shape = (V, D)

In [14]:
import torch.nn.functional as F

cat = torch.tensor([0.21, -0.43, 0.87, 0.15])
dog = torch.tensor([0.19, -0.40, 0.82, 0.18])
car = torch.tensor([-0.72, 0.31, 0.14, -0.55])

cat_dog = F.cosine_similarity(cat.unsqueeze(0), dog.unsqueeze(0))
cat_car = F.cosine_similarity(cat.unsqueeze(0), car.unsqueeze(0))

print("cat-dog:", cat_dog.item())
print("cat-car:", cat_car.item())

cat-dog: 0.999123215675354
cat-car: -0.2523055970668793


# 9. Positional Information

Consider:

"I love Python"

and:

"Python love I"

They contain the same tokens but have different meanings.

Therefore, the model needs information about the position of each token.

For example:

I       → position 0
love    → position 1
Python  → position 2

Modern Transformer-based models combine token information with
positional information.

We will study positional embeddings properly when we build the
Transformer architecture.

For this chapter, remember:

> Token embeddings tell the model WHAT the token is.
> Positional information tells the model WHERE the token occurs.

# 10. Complete Text Representation Pipeline

We can now understand the beginning of an LLM pipeline:

                    Raw Text
                       ↓
                  Tokenization
                       ↓
                     Tokens
                       ↓
                  Token IDs
                       ↓
                Token Embeddings
                       ↓
             Positional Information
                       ↓
              Neural Network / LLM